[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/tracing/openinference_best_practices_tutorial.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">OpenInference 最佳实践</h1>

本 Notebook 通过实际操作介绍使用 [Phoenix](https://phoenix.arize.com) 追踪 AI 应用的 OpenInference 最佳实践。你将学习：

- OpenInference 如何在 OpenTelemetry 之上添加 AI 语义
- 如何通过会话、追踪和 span 的层级关系组织遥测数据
- 捕获 span 的三种方式：自动埋点、手动埋点及二者结合的混合埋点
- 所有 span 的通用属性，以及 LLM、chain、agent 和 tool 四种核心类型各自的属性
- 如何添加或覆盖 span 属性，包括无法直接访问的自动埋点 span

每节运行一小段代码，然后引导你在 Phoenix 追踪视图中观察结果。

## 初始设置

本 Notebook 自包含：安装 Phoenix，在 Notebook 会话内启动，并将追踪发送给它。由于 Phoenix 运行于会话本身，在本地和 Google Colab 中运行方式相同，无需单独启动终端或服务器。

以下步骤安装库、设置 OpenAI API 密钥、启动 Phoenix 并配置追踪。

### 安装库

In [ ]:
!pip install -qq arize-phoenix arize-phoenix-otel openai openai-agents openinference-instrumentation-openai openinference-instrumentation-openai-agents

### 设置密钥

LLM 调用需要 OpenAI API 密钥。从 OpenAI 账号获取密钥，并在下方提示时输入。Phoenix 在本地运行，无需 Phoenix API 密钥。

In [ ]:
import os
from getpass import getpass

OPENAI_API_KEY = globals().get("OPENAI_API_KEY") or getpass("🔑 请输入 OpenAI API 密钥： ")
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

### 启动 Phoenix

直接在 Notebook 会话中启动 Phoenix 应用。服务器在后台线程中运行，因此本地与 Google Colab 的行为一致，无需单独启动终端或服务器。

运行下方单元，Phoenix 界面会**内嵌**显示在下方，尤其适合 Colab。本地运行时也可以在独立浏览器标签页打开输出的 `http://localhost:6006`。之后可以在任意单元再次调用 `session.view()`，将实时视图显示到当前工作位置。

In [ ]:
import phoenix as px

session = px.launch_app()
print(f"🔭 Phoenix 已运行，界面地址： {session.url}")

# 启用嵌套事件循环，使异步 Agents SDK 示例可以在 Notebook 中运行。
# 在启动 Phoenix 后再应用此设置，否则会导致 Phoenix 服务器
# 无法在 Python 3.12（Google Colab 使用的版本）上启动。
import nest_asyncio

nest_asyncio.apply()

# 内嵌显示 Phoenix 界面，这是在 Colab 中查看界面的可靠方式，因为
# 上方 URL 往往无法在新标签页打开。可在任意单元格重新运行 `session.view()`
# 以在当前工作位置显示实时界面。
session.view()

### 配置追踪

使用 `arize-phoenix-otel` 的便捷函数注册将 span 发送到 Phoenix 的追踪提供程序，再启用 OpenAI 自动埋点，以自动追踪 OpenAI SDK 调用。

完整的 `phoenix.otel` 函数及单应用向多个项目路由追踪的方法，见 [phoenix.otel 辅助函数](/docs/phoenix/tracing/concepts-tracing/otel-openinference/phoenix-otel-helpers)。

In [ ]:
from openinference.instrumentation.openai import OpenAIInstrumentor
from phoenix.otel import register

tracer_provider = register(
    project_name="otel-best-practices",
    batch=False,
)

OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

## OpenInference 简介

[OpenInference](https://github.com/Arize-ai/openinference) 是一组用于追踪 AI 应用的开源约定与埋点库，由 Arize 维护。Phoenix 按照此标准，在追踪视图中展示 LLM、工具、智能体、链、检索器及其他 AI 专用 span。

OpenInference 是 **[OpenTelemetry](https://opentelemetry.io/docs/) 的扩展**。底层使用标准 OpenTelemetry SDK 和库，包括其他 OTel 服务使用的 `TracerProvider`、`Tracer`、`Span`、`SpanProcessor` 和 `Exporter`。它添加了：

- **语义约定**：规定如何通过 span 属性表示 LLM 调用、提示词、消息、工具调用、检索、智能体步骤和会话等 AI 概念
- **自动埋点库**：覆盖 OpenAI、Anthropic、Bedrock、LangChain、LlamaIndex、CrewAI、AutoGen 等常用 LLM SDK 和编排框架

任何支持 OTel 的工具都可以接收这些 span；理解 OpenInference 约定的后端（如 Phoenix）还能够将其展示为丰富的 AI 追踪视图。

**延伸阅读：**

- [OpenTelemetry 与 OpenInference 概念](/docs/phoenix/tracing/concepts-tracing/otel-openinference/overview)：本教程的完整配套参考
- [OpenInference 语义约定规范](https://github.com/Arize-ai/openinference/tree/main/spec)：span 类型、属性和消息结构的正式定义
- [OpenInference 仓库](https://github.com/Arize-ai/openinference)：各支持语言和框架的自动埋点、示例和源码
- [OpenTelemetry 文档](https://opentelemetry.io/docs/)：OpenInference 底层使用的可观测性框架

下方代码模拟与聊天机器人对话并提出两个相关问题，通过简单的 OpenAI 调用在 Phoenix 中创建追踪。

先运行代码，后面再深入介绍细节。

In [ ]:
import uuid

from IPython.display import Markdown, display
from openai import OpenAI
from openinference.instrumentation import using_session
from openinference.semconv.trace import OpenInferenceSpanKindValues, SpanAttributes
from opentelemetry import trace

client = OpenAI()
session_id = str(uuid.uuid4())
tracer = trace.get_tracer(__name__)
messages = [
    {"role": "system", "content": "你是乐于助人的助手，请简洁回答。"}
]


def ask_llm(question: str) -> None:
    messages.append({"role": "user", "content": question})

    with tracer.start_as_current_span("openinference-intro-chain") as chain_span:
        chain_span.set_attribute(
            SpanAttributes.OPENINFERENCE_SPAN_KIND,
            OpenInferenceSpanKindValues.CHAIN.value,
        )
        chain_span.set_attribute(SpanAttributes.INPUT_VALUE, question)

        response = client.responses.create(
            model="gpt-5.4-mini",
            input=messages,
        )
        answer = response.output_text

        messages.append({"role": "assistant", "content": answer})
        chain_span.set_attribute(SpanAttributes.OUTPUT_VALUE, answer)
        display(Markdown(answer))


with using_session(session_id=session_id):
    ask_llm("What is OpenInference?")
    ask_llm("How does it relate to OpenTelemetry?")

打开 Phoenix 界面，使用上方启动时输出的 URL，进入 `otel-best-practices` 项目查看追踪。

### 会话、追踪与 span

OpenInference 和 OpenTelemetry 通过三个相互嵌套的概念组织追踪数据：

- **Span**：应用中的单个步骤，例如 LLM 调用、工具调用或链中的阶段。包含名称、起止时间、属性及可选的父 span。span 嵌套形成树，树根称为根 span，没有父节点。
- **Trace（追踪）**：共享同一个 `trace_id` 的 span 集合，表示通过智能体的一次端到端请求。
- **Session（会话）**：共享同一个 `session.id` 的追踪集合，按共同概念逻辑分组，例如完成同一任务的多次交互或连续对话。

以客服聊天机器人为例：用户的完整多轮对话构成一个**会话**；每轮用户消息及应用回答构成一条**追踪**；每条追踪内，意图分类、调用工具、格式化回答等步骤各自构成 **span**。

```
Session: support-chat-7a3f
│
├── Trace 1: "Where is my order?"
│   └── Chain span: handle_message
│       ├── LLM span:  classify_intent
│       ├── Tool span: lookup_order(order_id="A123")
│       └── LLM span:  format_response
│
├── Trace 2: "When will it arrive?"
│   └── Chain span: handle_message
│       ├── LLM span:  classify_intent
│       ├── Tool span: get_shipping_status(order_id="A123")
│       └── LLM span:  format_response
│
└── Trace 3: "Thanks!"
    └── Chain span: handle_message
        └── LLM span: generate_acknowledgement
```

三条追踪，一个会话。在 Phoenix 中可以打开单条追踪调试某轮交互，也可以通过会话视图统一查看所有轮次。

信号、span、追踪与会话的完整关系，见[信号、span、追踪与会话](/docs/phoenix/tracing/concepts-tracing/otel-openinference/signals)。

#### 会话

先打开 Phoenix 的 **Sessions** 标签页，应能看到一个表示完整两轮对话的会话。

![Phoenix Sessions 标签页中的聊天机器人会话](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/cookbooks/otel-cookbook/sessions.png)

![会话详情中的两轮对话输入输出、延迟、token 数和成本](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/cookbooks/otel-cookbook/session.png)

选择会话后，详情面板展示对话中的两个步骤、智能体输入输出、运行延迟、总 token 用量，以及依据 LLM 提供商公开单价估算的成本（若可获取）。

#### 追踪

选择 **Traces** 标签页，可以看到对话中的两个步骤分别显示为独立追踪，各自展示智能体输入输出。

示例仅调用一次 LLM，因此追踪输入是发给 LLM 的内容，输出是 LLM 的回答。更复杂的追踪中，输入是传给智能体的内容，输出则是智能体完成全部 LLM 和工具调用后给出的最终回答。

![Phoenix Traces 标签页中的两条对话追踪](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/cookbooks/otel-cookbook/traces.png)

![单条追踪详情中的 span 树、延迟、token 数和成本](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/cookbooks/otel-cookbook/trace.png)

选择追踪后，详情面板展示 span 树、延迟、token 数和估算成本。

> 也可以从会话视图直接进入各条追踪。

#### Span

追踪视图用树状结构展示组成追踪的 span。具有同一 `trace_id` 的 span 归属于同一追踪。树顶部是根 span，其余节点位于下方。层级通过 `parent_id` 定义：每个子 span 的 `parent_id` 指向其父 span 的 ID。

本例追踪包含两个 span：

![单条追踪详情中的 span 树、延迟、token 数和成本](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/cookbooks/otel-cookbook/trace.png)

![追踪树中选中的 ChatCompletion LLM span](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/cookbooks/otel-cookbook/llm-span.png)

![所选 span 在 Attributes 标签页中的 JSON 属性](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/cookbooks/otel-cookbook/span-attributes.png)

根节点是 **Chain** span，是一组相关 span 的起点，可视作组织 span 的文件夹。本例其实不必创建 chain span，只是为了演示树状结构。

根节点下方名为 `ChatCompletion` 的 **LLM** span 表示一次 OpenAI LLM 调用。

各 span 的 **Attributes** 标签页以 JSON 展示相关属性，包括输入输出、LLM token 用量等，后续章节会进一步介绍。

OpenInference 定义了固定的 span 类型：

| Span 类型 | 说明 |
| --- | --- |
| `LLM` | 大语言模型调用，捕获模型、输入输出消息、token 数和成本。 |
| `CHAIN` | 应用步骤的起点或连接，常用作父 span，将相关工作组织为逻辑块。 |
| `AGENT` | 表示智能体工作，通常包裹 LLM 和工具 span。 |
| `TOOL` | 外部工具或函数调用，常由 LLM 的工具使用请求触发。 |
| `RETRIEVER` | 检索操作，例如从向量库或搜索索引获取文档。 |
| `EMBEDDING` | 嵌入模型调用。 |
| `RERANKER` | 对检索结果重新排序的步骤。 |
| `GUARDRAIL` | 安全或策略检查，例如内容审核、个人身份信息检测或输入验证。 |
| `EVALUATOR` | 为 LLM 输出评分或判断的评估步骤。 |
| `PROMPT` | 提示词定义或模板化步骤。 |
| `UNKNOWN` | 无其他适用类型时使用。 |

本教程关注 LLM、chain、agent 和 tool。所有类型及其预期属性的完整参考，见 [OpenInference span 类型](/docs/phoenix/tracing/concepts-tracing/otel-openinference/span-kinds)。

## 配置会话

**会话**按照与智能体的连续交互对追踪进行逻辑分组。例如，单个用户与聊天机器人的完整多轮对话构成一个会话；该用户开始没有历史上下文的新对话，或另一用户开始对话时，应创建新会话。

会话由开发智能体的工程师显式管理，发送追踪不会自动创建会话。

使用 `using_session` 设置会话 ID，作用于代码块内创建的所有 span。它属于 OpenInference 上下文管理器系列，其他成员包括 `using_user`、`using_metadata`、`using_tags`、`using_prompt_template` 和 `using_attributes`，完整列表见 [OpenInference 上下文管理器](/docs/phoenix/tracing/concepts-tracing/otel-openinference/context-managers)。

下面在会话内调用 OpenAI。由于会话 ID 写死，多次运行会在同一会话内创建新追踪。

In [ ]:
with using_session(session_id="My Session"):
    response = client.responses.create(
        model="gpt-5.4-mini",
        input="What are sessions in OpenInference? Be concise.",
    )
    display(Markdown(response.output_text))

在 Phoenix 中查找此会话。根据运行次数，会看到一个会话内包含多条追踪。

![包含多条追踪的单个会话](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/cookbooks/otel-cookbook/sessions-1-session.png)

再运行下方代码。它使用不同的会话 ID，因此会创建新会话。

In [ ]:
with using_session(session_id="My Session 2"):
    response = client.responses.create(
        model="gpt-5.4-mini",
        input="What are sessions in OpenInference? Be concise.",
    )
    display(Markdown(response.output_text))

会话列表中现在会出现新会话。

![Sessions 列表中的两个独立会话](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/cookbooks/otel-cookbook/sessions-2-sessions.png)

## 捕获 span 与追踪

前面的示例已经展示 OpenInference 创建 span 的两种方式：

- **自动埋点**：包装 OpenAI SDK、LangChain 等特定库或框架，自动为每次调用及工具调用等框架动作创建 span。每次库或框架调用是独立追踪。
- **手动埋点**：直接在应用代码中调用追踪器，创建自己的追踪和 span。

实际应用通常结合两者：自动埋点负责标准 SDK 调用，手动埋点捕获包裹这些调用的应用逻辑。

更深入的比较见[埋点方式](/docs/phoenix/tracing/concepts-tracing/otel-openinference/instrumentation-approaches)。

### 自动埋点

自动埋点库为 SDK 或框架添加埋点，自动为每次调用和执行的动作创建 span。

初始设置中已配置 OpenAI 自动埋点：

```python
from openinference.instrumentation.openai import OpenAIInstrumentor

OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)
```

执行这一次配置后，代码中的每个 `client.responses.create()` 和 `client.chat.completions.create()` 都会在新追踪中自动创建 LLM span。若框架支持工具调用，每次框架调用会创建新追踪，LLM 和工具调用的 span 归于 chain span 下。

OpenInference 支持 OpenAI、Anthropic、Bedrock、LangChain、LlamaIndex、CrewAI、AutoGen 等常用 SDK 和编排框架，完整列表见 [OpenInference 仓库](https://github.com/Arize-ai/openinference)。

运行下方代码，自动埋点将创建只包含一个 span 的追踪，可在 Phoenix 中查看。

In [ ]:
with using_session(session_id="Capturing Spans Example"):
    response = client.responses.create(
        model="gpt-5.4-mini",
        input="What are sessions in OpenInference? Be concise.",
    )
    display(Markdown(response.output_text))

在 Phoenix 的 `Capturing Spans Example` 会话中打开新追踪，会看到一个 LLM span。它由自动埋点针对 `client.responses.create()` 创建，无需手写追踪代码。

### 手动埋点

手动埋点提供完整控制。直接调用 OpenTelemetry 追踪器启动 span、设置属性（包括 OpenInference 类型），并在工作完成后结束。推荐使用上下文管理器，将 span 设为 OpenTelemetry 上下文中的活动 span，这样代码块内创建的 span 自动成为其子节点，退出时自动结束：

```python
with tracer.start_as_current_span("my-span") as span:
    span.set_attribute(
        SpanAttributes.OPENINFERENCE_SPAN_KIND,
        OpenInferenceSpanKindValues.CHAIN.value,
    )
    span.set_attribute(SpanAttributes.INPUT_VALUE, "input data")
    # 在此执行工作
    span.set_attribute(SpanAttributes.OUTPUT_VALUE, "result")
```

设置 `openinference.span.kind` 属性即可手动创建任意 OpenInference 类型的 span，例如 chain、LLM 或 tool。类型决定 Phoenix 使用的图标和详情视图，以及 span 预期具有的属性集合。

下方代码创建包含三个手动 span 的追踪：父级 `data-pipeline` chain span 内嵌两个子 span（`step-1-validate` 与 `step-2-format`）。没有 LLM 或工具调用，所有 span 均由代码创建。

In [ ]:
with using_session(session_id="Capturing Spans Example"):
    with tracer.start_as_current_span("data-pipeline") as pipeline:
        pipeline.set_attribute(
            SpanAttributes.OPENINFERENCE_SPAN_KIND,
            OpenInferenceSpanKindValues.CHAIN.value,
        )
        pipeline.set_attribute(SpanAttributes.INPUT_VALUE, "raw request")

        with tracer.start_as_current_span("step-1-validate") as step:
            step.set_attribute(
                SpanAttributes.OPENINFERENCE_SPAN_KIND,
                OpenInferenceSpanKindValues.CHAIN.value,
            )
            step.set_attribute(SpanAttributes.INPUT_VALUE, "raw request")
            step.set_attribute(SpanAttributes.OUTPUT_VALUE, "validated request")

        with tracer.start_as_current_span("step-2-format") as step:
            step.set_attribute(
                SpanAttributes.OPENINFERENCE_SPAN_KIND,
                OpenInferenceSpanKindValues.CHAIN.value,
            )
            step.set_attribute(SpanAttributes.INPUT_VALUE, "validated request")
            step.set_attribute(SpanAttributes.OUTPUT_VALUE, "formatted output")

        pipeline.set_attribute(SpanAttributes.OUTPUT_VALUE, "formatted output")

在 Phoenix 的 `Capturing Spans Example` 会话中打开新追踪，可看到一个名为 `data-pipeline` 的 chain span，其下嵌套两个子 chain span：`step-1-validate` 和 `step-2-format`。

![data-pipeline span 内嵌套 step-1-validate 和 step-2-format 子 span](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/cookbooks/otel-cookbook/data-pipeline.png)

### 混合埋点

**混合埋点**结合两种方式，将自动埋点调用包裹在手动 span 中。这样既能将 SDK 调用组织成逻辑单元、添加自定义属性、构建符合应用结构的追踪树，也能保留自动埋点捕获的丰富属性。

二者共享 OpenTelemetry 上下文，因此可以自然嵌套。使用 `tracer.start_as_current_span(...)` 打开手动 span 后，它成为上下文中的活动 span；代码块内自动埋点 SDK 的调用会创建它的子 span。

OpenInference 简介中的 `ask_llm` 已采用此方式：每次 OpenAI 调用都包裹在手动创建的 chain span 内。

手动 chain span 是父节点，OpenAI 自动埋点围绕 `client.responses.create()` 创建的 LLM span 自动成为子节点。这正是前面在 Phoenix 中看到的树状结构：顶部为 chain span，内部嵌套 LLM span。

实际应用通常采用这种结构：手动 chain 或 agent span 展示业务逻辑的高层结构，自动埋点 span 补充每次 SDK 调用的底层细节。

运行以下代码查看示例：

In [ ]:
with using_session(session_id="Capturing Spans Example"):
    with tracer.start_as_current_span("manual-chain") as chain_span:
        chain_span.set_attribute(
            SpanAttributes.OPENINFERENCE_SPAN_KIND,
            OpenInferenceSpanKindValues.CHAIN.value,
        )
        chain_span.set_attribute(
            SpanAttributes.INPUT_VALUE, "What are sessions in OpenInference? Be concise."
        )

        response = client.responses.create(
            model="gpt-5.4-mini",
            input="What are sessions in OpenInference? Be concise.",
        )

        chain_span.set_attribute(SpanAttributes.OUTPUT_VALUE, response.output_text)
        display(Markdown(response.output_text))

在 Phoenix 的 `Capturing Spans Example` 会话中打开新追踪，将看到名为 `manual-chain` 的 chain span，内部嵌套一个 LLM span。由于共享 OpenTelemetry 上下文，手动 span 是父节点，自动埋点 LLM span 自动成为子节点。

## Span 属性

每个 OpenInference span 都可以携带与类型无关的**通用属性**，并叠加**类型专用属性**。

所有 span 类型可用的通用属性如下：

| 属性 | 说明 |
| --- | --- |
| `openinference.span.kind` | 类型为 `LLM`、`CHAIN`、`AGENT`、`TOOL`、`RETRIEVER`、`EMBEDDING`、`RERANKER`、`GUARDRAIL`、`EVALUATOR`、`PROMPT` 或 `UNKNOWN`，决定 Phoenix 如何展示。 |
| `input.value` | 字符串形式的输入。结构化值应序列化为 JSON，并相应设置 `input.mime_type`。 |
| `input.mime_type` | `input.value` 的 MIME 类型，默认为 `text/plain`；JSON 字符串应设置为 `application/json`。 |
| `output.value` | 字符串形式的输出。 |
| `output.mime_type` | `output.value` 的 MIME 类型，遵循与输入相同的约定。 |
| `metadata` | 包含自定义字段的 JSON 字典，用于附加用户等级、功能开关、请求 ID 等领域上下文。 |
| `session.id` | 将多条追踪归入会话，通过 `using_session(...)` 或 `span.set_attribute(SpanAttributes.SESSION_ID, ...)` 设置。 |
| `user.id` | 追踪所属用户的标识，可通过 `using_user(...)` 或直接设置。 |
| `tag.tags` | 用于筛选的字符串标签列表，通过 `using_tags(...)` 设置。 |

Phoenix 界面直接使用其中多项属性：`openinference.span.kind` 决定图标和类型详情视图；根 span 的 `input.value` 和 `output.value` 用于 Traces 与 Sessions 页的输入输出预览；`session.id` 组织会话；`metadata` 和 `tag.tags` 支持跨 span 筛选。

完整属性目录见 [OpenInference 语义约定](/docs/phoenix/tracing/concepts-tracing/otel-openinference/semantic-conventions)（整体标准）和 [OpenInference span 类型](/docs/phoenix/tracing/concepts-tracing/otel-openinference/span-kinds)（各类型参考）。

## 核心 span 类型

OpenInference 定义 11 种 span 类型，但多数 AI 应用主要使用 **LLM**、**chain**、**agent** 和 **tool** 四种。智能体发起 LLM 调用，LLM 触发工具调用，chain span 将相关工作分组。完整类型及属性的权威参考见 [OpenInference span 类型](/docs/phoenix/tracing/concepts-tracing/otel-openinference/span-kinds)。

下面使用 [OpenAI Agents SDK](https://github.com/openai/openai-agents-python) 生成包含全部四种类型的单条追踪。其专用自动埋点库 `openinference-instrumentation-openai-agents` 会为每次智能体运行生成完整的 span 类型集合。

将 Agents SDK 埋点连接到现有追踪提供程序。可以同时保留 OpenAI 自动埋点：Agents SDK 创建自己的 LLM span，二者可以协同工作。

In [ ]:
from openinference.instrumentation.openai_agents import OpenAIAgentsInstrumentor

OpenAIAgentsInstrumentor().instrument(tracer_provider=tracer_provider)

定义具有两个工具的简单旅行助手，再提出需要同时调用这两个工具的问题。将运行过程包裹在会话中，便于在 Phoenix 中查找。

In [ ]:
from agents import Agent, Runner, function_tool


@function_tool
def get_weather(city: str) -> str:
    """获取指定城市的当前天气。"""
    return f"The weather in {city} is sunny and 22°C."


@function_tool
def get_time_zone(city: str) -> str:
    """获取指定城市的时区。"""
    zones = {
        "Tokyo": "JST (UTC+9)",
        "London": "GMT (UTC+0)",
        "New York": "EST (UTC-5)",
    }
    return zones.get(city, "unknown")


travel_agent = Agent(
    name="TravelAssistant",
    instructions=(
        "你是乐于助人的旅行助手。"
        "使用 get_weather 查询城市天气，"
        "使用 get_time_zone 查询时区。"
        "最后简洁回答。"
    ),
    tools=[get_weather, get_time_zone],
)

with using_session(session_id="Travel Agent Example"):
    result = await Runner.run(
        travel_agent,
        "What's the weather and time zone in Tokyo?",
    )

display(Markdown(result.final_output))

在 Phoenix 的 `otel-best-practices` 项目中打开追踪。一次智能体运行产生的追踪包含四种核心 span：

- 两个 **agent** span：外层 `Agent workflow` 包装与内层 `TravelAssistant`
- 三个 **chain** span：一个包装完整工作流，另两个分别对应每轮智能体交互
- 两个 **tool** span：分别对应 `get_weather` 和 `get_time_zone` 调用
- 四个 **LLM** span：每轮都会产生 SDK 的 Responses API 调用，内部嵌套底层 OpenAI 客户端调用

![OpenAI Agents SDK 追踪中嵌套的 agent、chain、tool 和 LLM span](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/cookbooks/otel-cookbook/agent-trace.png)

多数复杂智能体的追踪都采用这种结构：外层 agent/chain 工作流、每次外部调用的 tool span，以及每次模型调用的 LLM span。

### LLM span

**LLM span** 表示一次大语言模型调用，捕获调试或分析所需的模型、输入消息、输出、工具、token 数、成本等信息。

选择智能体追踪中的任意 LLM span，可以查看该调用的输入输出。树中还会展示 token 用量、延迟和成本，**Attributes** 标签页包含完整属性。

![LLM span 的完整属性](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/cookbooks/otel-cookbook/llm-span-attributes.png)

本示例的相关属性如下：

```json
{
    "openinference": {
        "span": {
            "kind": "LLM"
        }
    },
    "llm": {
        "cost": {
            "completion": 0.0002205,
            "completion_details": {
                "output": 0.0002205,
                "reasoning": 0
            },
            "prompt": 0.00009975,
            "prompt_details": {
                "cache_read": 0,
                "input": 0.00009975
            },
            "total": 0.00032025
        },
        "input_messages": [
            {
                "message.content": "You are a helpful travel assistant. Use get_weather to look up the weather for a city, and get_time_zone to look up its time zone. Give a concise final answer.",
                "message.role": "system"
            },
            {
                "message.content": "What's the weather and time zone in Tokyo?",
                "message.role": "user"
            }
        ],
        "invocation_parameters": "{\"include\": [], \"model\": \"gpt-5.4-mini\", \"prompt_cache_key\": \"agents-sdk:run:1fc4521fe6f248beafa82586c8b0fa3e\", \"reasoning\": {\"effort\": \"none\"}, \"text\": {\"verbosity\": \"low\"}}",
        "model_name": "gpt-5.4-mini-2026-03-17",
        "output_messages": [
            {
                "message.role": "assistant",
                "message.tool_calls": [
                    {
                        "tool_call.function.arguments": "{\"city\":\"Tokyo\"}",
                        "tool_call.function.name": "get_weather",
                        "tool_call.id": "call_OdvgB0fVwTcdKy6mqxV3cmuB"
                    }
                ]
            },
            {
                "message.role": "assistant",
                "message.tool_calls": [
                    {
                        "tool_call.function.arguments": "{\"city\":\"Tokyo\"}",
                        "tool_call.function.name": "get_time_zone",
                        "tool_call.id": "call_cnCgrd4Js5bErfROFdIoc68j"
                    }
                ]
            }
        ],
        "provider": "openai",
        "system": "openai",
        "token_count": {
            "completion": "49",
            "completion_details": {
                "output": "49",
                "reasoning": "0"
            },
            "prompt": "133",
            "prompt_details": {
                "cache_read": "0",
                "input": "133"
            },
            "total": "182"
        },
        "tools": [
            {
                "tool.json_schema": "{\"name\":\"get_weather\",\"parameters\":{\"properties\":{\"city\":{\"title\":\"City\",\"type\":\"string\"}},\"required\":[\"city\"],\"title\":\"get_weather_args\",\"type\":\"object\",\"additionalProperties\":false},\"strict\":true,\"type\":\"function\",\"defer_loading\":null,\"description\":\"Get the current weather for a city.\"}"
            },
            {
                "tool.json_schema": "{\"name\":\"get_time_zone\",\"parameters\":{\"properties\":{\"city\":{\"title\":\"City\",\"type\":\"string\"}},\"required\":[\"city\"],\"title\":\"get_time_zone_args\",\"type\":\"object\",\"additionalProperties\":false},\"strict\":true,\"type\":\"function\",\"defer_loading\":null,\"description\":\"Get the time zone for a city.\"}"
            }
        ]
    }
}
```

#### LLM 专用属性

除通用属性外，OpenAI 自动埋点还为每个 LLM span 添加符合 OpenInference 语义约定的 `llm.*` 属性：

| 属性 | 说明 |
| --- | --- |
| `llm.model_name` | OpenAI 返回的精确模型标识，例如 `gpt-5.4-mini-2026-03-17`，是解析后的快照版本，并非传入的别名。 |
| `llm.provider` | LLM 提供商，此处为 `openai`。 |
| `llm.system` | AI 系统标识，此处同样为 `openai`。 |
| `llm.invocation_parameters` | 传给 API 的参数 JSON 字符串，例如 `{"model": "gpt-5.4-mini"}`。 |
| `llm.input_messages` | 发送给 API 的结构化消息数组，每项包含 `message.role` 和 `message.content`。 |
| `llm.output_messages` | API 返回的结构化消息数组，每项包含 `message.role` 及由结构化内容项组成的 `message.contents` 列表，内容项具有 `message_content.text` 与 `message_content.type`。此结构兼容文本、图片、音频和推理等多模态内容。 |
| `llm.token_count.prompt` / `.completion` / `.total` | 调用的 token 数，详细分类见 `llm.token_count.prompt_details.*`（`cache_read`、`input`）和 `llm.token_count.completion_details.*`（`output`、`reasoning`）。 |
| `llm.cost.prompt` / `.completion` / `.total` | 以美元计算的估算成本，具有相同的 `_details` 分类。Phoenix 根据 token 用量和模型公开价格计算。 |

这些属性驱动追踪和会话视图中的 token 数与成本列，并填充 LLM 详情视图的输入消息、输出消息及模型名称。

### 工具 span

**工具 span** 表示对外部工具或函数的调用，通常由 LLM 决定发起。它记录工具名称、LLM 传入的参数以及工具返回值。

上方追踪具有 `get_weather` 和 `get_time_zone` 两个工具 span。选择任意一个，可以查看工具名称、传入参数 `{"city": "Tokyo"}` 和返回值等属性。

![get_weather 工具 span 的输入城市和返回天气文本](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/cookbooks/otel-cookbook/tool-span-attributes.png)

本示例的相关属性如下：

```json
{
    "openinference": {
        "span": {
            "kind": "TOOL"
        }
    },
    "tool": {
        "name": "get_weather"
    }
}
```

#### 工具专用属性

除通用属性外，工具 span 还包含 `tool.*` 命名空间中的少量专用属性：

| 属性 | 说明 |
| --- | --- |
| `tool.id` | 工具调用结果的标识，对应 LLM 生成的 `tool_call.id`，使 Phoenix 能关联回请求此工具的 LLM 调用。 |
| `tool.name` | 工具名称。 |
| `tool.description` | 工具描述，LLM 据此决定调用哪个工具。 |
| `tool.parameters` | LLM 传入工具的参数值 JSON 字符串。 |
| `tool.json_schema` | 工具输入的完整 JSON schema，通常采用 OpenAI 工具调用格式，说明工具期望的参数结构。 |

Phoenix 在工具详情视图顶部展示名称和描述，从 `input.value` 与 `output.value` 展示参数和返回值，并通过 `tool.id` 将工具 span 关联到父级 LLM span。

### 智能体 span

**智能体 span** 表示自主智能体的编排工作：决定何时调用 LLM、调用工具、再次调用 LLM，以及何时停止。它通常是组成智能体循环的 LLM、工具和 chain span 的父节点。

上方追踪中的 `TravelAssistant` 是智能体 span。选择它可以查看两轮交互，以及其中嵌套的全部 LLM 和工具调用。

![TravelAssistant 智能体 span 的属性](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/cookbooks/otel-cookbook/agent-span-attributes.png)

本示例的相关属性如下：

```json
{
    "openinference": {
        "span": {
            "kind": "AGENT"
        }
    },
    "graph": {
        "node": {
            "id": "TravelAssistant"
        }
    }
}
```

注意，OpenAI Agents SDK 自动埋点使用 `graph.node.id` 保存智能体名称 `TravelAssistant`，而非约定中的 `agent.name`，便于 Phoenix 将带有交接的多智能体系统展示为图视图。手动创建智能体 span 时，应设置 `agent.name`，如需图视图，也可设置 `graph.node.*`。

#### 智能体专用属性

智能体 span 包含以下专用属性：

| 属性 | 说明 |
| --- | --- |
| `agent.name` | 智能体名称。承担相同逻辑角色的智能体应共享名称，便于将追踪分组。 |
| `graph.node.id` | 执行图中此智能体节点的 ID，可选；希望在 Phoenix 图视图中展示多智能体系统时设置。 |
| `graph.node.name` | 便于人阅读的图节点名称。 |
| `graph.node.parent_id` | 父节点 ID，根智能体不设置此值。 |

Phoenix 通过 `graph.node.*` 属性，在追踪树之外将 LangGraph、AutoGen、CrewAI 等多智能体系统展示为图视图。

### Chain span

**Chain span** 用于通用分组。需要在追踪树中将相关工作组织到同一父节点时使用，例如多步骤流水线、单轮智能体交互、包含前后处理的 LLM 调用，或应用代码中的逻辑块。

上方智能体追踪中，外层 `Agent workflow` 是 chain span，每轮交互也包裹在名为 `turn` 的 chain span 中。选择 `turn` 可查看它如何组织该轮 LLM 与工具调用。

![turn chain span 组织同一轮的 LLM 和工具调用](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/cookbooks/otel-cookbook/chain-span-attributes.png)

本示例的相关属性如下：

```json
{
    "openinference": {
        "span": {
            "kind": "CHAIN"
        }
    },
}
```

Chain span 仅携带类型和通用属性，其价值在于结构：在追踪树中提供一个具名父节点。

#### Chain 专用属性

Chain span 没有类型专用属性，依赖前述通用属性，包括 `openinference.span.kind`、`input.value`、`output.value`、`metadata` 和 `session.id` 等。它提供具名父节点来组织追踪树，可以按需附加输入输出。

## 覆盖或添加属性

自动埋点会为创建的每个 span 捕获标准 OpenInference 属性，但通常还需要添加自定义属性，例如：

- **添加筛选标签**：如 `experiment="v2-prompt"` 或 `tenant="acme"`
- **附加领域元数据**：用户等级、请求 ID、功能开关值
- **记录提示词模板**：独立保存模板字符串、版本和变量，而非只记录展开后的最终提示词

手动 span 可以在 `with` 块内直接调用 `span.set_attribute(...)`。自动埋点 span 无法直接访问，但可以通过 **OpenInference 上下文管理器**把属性放入 OpenTelemetry 上下文。自动埋点创建 span 时会读取这些属性，因此块内所有 span 都会携带属性，无论由谁创建。

可用的上下文管理器如下：

| 上下文管理器 | 设置的属性 |
| --- | --- |
| `using_session(session_id)` | `session.id` |
| `using_user(user_id)` | `user.id` |
| `using_metadata(metadata)` | `metadata`，包含自定义字段的 JSON 字典 |
| `using_tags(tags)` | `tag.tags`，字符串列表 |
| `using_prompt_template(template=, variables=, version=)` | `llm.prompt_template.*`，对 LLM span 尤其有用 |
| `using_attributes(...)` | 将上述设置合并为一次调用的便捷包装 |

完整用法及注意事项见 [OpenInference 上下文管理器](/docs/phoenix/tracing/concepts-tracing/otel-openinference/context-managers)。

下方使用 `using_metadata` 附加领域元数据字典。本例包裹 OpenAI 调用，使元数据出现在 LLM span 上；此方式同样适用于块内创建的任何自动或手动 span。

In [ ]:
from openinference.instrumentation import using_metadata

with using_session(session_id="LLM Span Example"):
    with using_metadata({"user_tier": "premium", "request_source": "cookbook"}):
        response = client.responses.create(
            model="gpt-5.4-mini",
            input="What are LLM spans in OpenInference? Be concise.",
        )
        display(Markdown(response.output_text))

在 Phoenix 中打开新追踪，LLM span 现在多出一个属性：

- `metadata`：包含 `{"user_tier": "premium", "request_source": "cookbook"}` 的 JSON 字符串

它在 **Attributes** 标签页中与标准 `llm.*` 属性一起显示。

![LLM span 的自定义 metadata 和标准 llm.* 属性](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/cookbooks/otel-cookbook/llm-span-metadata.png)

![按 attributes.metadata.request_source 等于 cookbook 筛选的 Spans 标签页](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/cookbooks/otel-cookbook/metadata-filter.png)

还可以按 `metadata` 值筛选，按租户、功能开关或其他领域维度切分追踪。在 **Spans** 标签页将筛选条件设为 `attributes.metadata.request_source = "cookbook"`，只显示 `request_source` 元数据为 `cookbook` 的 span。

### 覆盖工具属性

也可以覆盖自动埋点设置的属性，或补充遗漏属性。OpenAI Agents SDK 自动埋点仅为工具 span 设置 `tool.name`，不会填充 `tool.description`。下方重新定义 `get_weather`，在活动工具 span 上设置自定义 `tool.description`，然后重建并运行智能体。

自动埋点在调用函数前就打开了工具 span，因此函数体执行时，工具 span 已是活动 span。在函数内调用 `set_attribute(...)`，如果属性已存在就覆盖，否则添加。

In [ ]:
@function_tool
def get_weather(city: str) -> str:
    """获取指定城市的当前天气。"""

    # 在活动工具跨度上设置自定义 tool.description。
    trace.get_current_span().set_attribute(
        SpanAttributes.TOOL_DESCRIPTION,
        "查询给定城市的当前天气状况。",
    )

    return f"The weather in {city} is sunny and 22°C."


travel_agent = Agent(
    name="TravelAssistant",
    instructions=(
        "你是乐于助人的旅行助手。"
        "使用 get_weather 查询城市天气，"
        "使用 get_time_zone 查询时区。"
        "最后简洁回答。"
    ),
    tools=[get_weather, get_time_zone],
)

with using_session(session_id="Tool Override Example"):
    result = await Runner.run(
        travel_agent,
        "What's the weather and time zone in Tokyo?",
    )

display(Markdown(result.final_output))

通过 `Tool Override Example` 会话，在 `otel-best-practices` 项目中打开新追踪。选择 `get_weather` 工具 span 并打开 **Attributes** 标签页，将看到函数体设置的 `tool.description`，以及自动埋点生成的标准 `tool.name`。

![get_weather 工具 span 的 tool.name 和手动设置的 tool.description](https://storage.googleapis.com/arize-phoenix-assets/assets/images/phoenix-docs-images/cookbooks/otel-cookbook/tool-span-attributes-override.png)

本示例的相关属性如下：

```json
{
    "openinference": {
        "span": {
            "kind": "TOOL"
        }
    },
    "tool": {
        "description": "Looks up the current weather conditions for a given city.",
        "name": "get_weather"
    }
}
```

同一追踪中的 `get_time_zone` 工具 span 仍只有 `tool.name`，直观说明覆盖仅作用于设置属性的那个 span。

## 总结

你已经了解使用 OpenInference 和 Phoenix 追踪 AI 应用的基本组成：

- **OpenInference 构建于 OpenTelemetry 之上**，为 AI 概念添加语义约定和自动埋点。底层仍由标准 OpenTelemetry SDK 驱动，`TracerProvider`、`Tracer`、`Span`、`SpanProcessor` 和 `Exporter` 保持不变。
- **Span、追踪和会话形成层级。** Span 是一个步骤，追踪是通过 `trace_id` 关联的 span 树，会话是通过 `session.id` 关联的一组追踪，可在 Phoenix 中串起多轮对话。
- **捕获 span 有三种方式。** 自动埋点包装 SDK，为每次调用生成 span；手动埋点通过 `tracer.start_as_current_span(...)` 创建 span；混合埋点以手动 span 包裹自动调用，形成追踪树的父节点。
- **所有类型都可携带通用属性**：`openinference.span.kind`、`input.value`、`output.value`、`metadata`、`session.id`、`user.id` 和 `tag.tags`。其中 `openinference.span.kind` 决定 Phoenix 的展示方式。
- **四种核心类型覆盖大部分 AI 应用**：LLM、chain、agent 和 tool。`llm.*` 保存模型名称、token 数和成本；`tool.*` 保存工具名称和描述；`agent.name` 或多智能体图的 `graph.node.id` 标识智能体；chain span 仅依赖通用属性。
- **可以丰富自动埋点 span。** 使用 `using_session`、`using_metadata`、`using_tags` 和 `using_prompt_template` 等上下文管理器附加属性，自动埋点通过 OpenTelemetry 上下文读取。也可以在工具函数内取得活动 span，直接调用 `set_attribute(...)` 覆盖或添加特定属性。

### 后续学习

- 阅读 Arize 文档的 [OpenTelemetry 与 OpenInference 概念](/docs/phoenix/tracing/concepts-tracing/otel-openinference/overview)，获取本教程的完整配套参考
- 查看 [OpenInference span 类型](/docs/phoenix/tracing/concepts-tracing/otel-openinference/span-kinds)，了解所有类型及其属性
- 阅读[埋点方式](/docs/phoenix/tracing/concepts-tracing/otel-openinference/instrumentation-approaches)，深入比较自动、手动与混合埋点
- 应用跨进程时，学习如何[跨服务或异步边界传播上下文](/docs/phoenix/tracing/concepts-tracing/otel-openinference/context-propagation)
- 使用[采样](/docs/phoenix/tracing/concepts-tracing/otel-openinference/sampling)降低生产环境追踪量
- 浏览 [OpenInference 仓库](https://github.com/Arize-ai/openinference)，了解 OpenAI、Anthropic、Bedrock、LangChain、LlamaIndex、CrewAI、AutoGen 等自动埋点
- 阅读 [OpenInference 语义约定规范](https://github.com/Arize-ai/openinference/tree/main/spec)，查看权威属性定义